# Модуль 7. Безопасность, совместимость и альтернативы

## Зачем этот модуль

Мы прошли путь от простой сериализации до полноценного backend-сервиса. Вы умеете сохранять модели, кешировать функции, распараллеливать вычисления и деплоить в API. Но в production есть ещё три важных темы, которые нельзя игнорировать:

1. **Безопасность** — загрузка чужого файла `.joblib` может сломать сервер или украсть данные.
2. **Совместимость** — модель, сохранённая сегодня, может не открыться через полгода.
3. **Альтернативы** — `joblib` — не единственный инструмент. Иногда лучше использовать `ONNX`, `safetensors` или просто JSON.

Этот модуль — страховка от ошибок в production и руководство по выбору инструмента под задачу.

## 1. Безопасность: почему `joblib.load()` — это как запуск незнакомой программы

### Аналогия с конвертом

Представьте, что вам передали запечатанный конверт и сказали: «Там внутри инструкции. Просто открой и выполни». Вы открываете, читаете, делаете всё, что написано. Но вдруг там написано: «Сотри все файлы на компьютере». Вы выполнили, потому что «открыть и выполнить» — это именно то, что делает `joblib.load()`.

### Что происходит при загрузке

`joblib.load()` (как и `pickle.load()`) не просто копирует данные. Он **восстанавливает объекты**, и в процессе может выполнить **произвольный Python-код**, который был встроен в файл при сохранении.

Это называется **десериализационная уязвимость** (deserialization vulnerability).

### Пример атаки (для понимания, не для использования)

In [1]:
import joblib

# Злоумышленник создаёт такой класс и сохраняет его
class Malicious:
    def __reduce__(self):
        import os
        return (os.system, ("echo 'Ваш сервер взломан' > /tmp/hacked.txt",))

joblib.dump(Malicious(), 'evil.joblib')

# Жертва загружает
obj = joblib.load('evil.joblib')  # Выполняется код!

При загрузке `evil.joblib` Python выполнит `os.system(...)` — и на диске появится файл `/tmp/hacked.txt`. В реальной атаке это может быть кража данных, удаление файлов или установка вируса.

### Правила безопасности

| Правило | Почему |
|---------|--------|
| **Никогда не загружайте `.joblib` из интернета** | Файл могут подменить |
| **Никогда не загружайте `.joblib` от неизвестных пользователей** | Это прямой путь к удалённому выполнению кода |
| **Храните модели в защищённом хранилище** | S3 с доступом только для вашего сервера, закрытый Git LFS |
| **Проверяйте хеш файла перед загрузкой** | SHA-256 или MD5: если файл изменился — не загружай |
| **Используйте read-only контейнеры** | Даже если код выполнится, он не сможет ничего записать |

### Проверка целостности файла

In [ ]:
import hashlib

def file_hash(path):
    """Вычисляет SHA-256 хеш файла."""
    sha256 = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            sha256.update(chunk)
    return sha256.hexdigest()

# Сохраняем хеш рядом с моделью
model_hash = file_hash('model.joblib')
with open('model.joblib.sha256', 'w') as f:
    f.write(model_hash)

# При загрузке проверяем
expected = open('model.joblib.sha256').read().strip()
actual = file_hash('model.joblib')

if expected != actual:
    raise SecurityError("Файл модели был изменён!")

## 2. Совместимость: модель из прошлого

### Почему `.joblib` — это не вечный формат

Файл `.joblib` — это «замороженный» Python-объект. Чтобы его «разморозить», Python должен:
- понять формат сериализации;
- найти все классы, которые использовались при сохранении;
- восстановить структуру объектов так, как она была в момент сохранения.

Если за это время:
- обновился `sklearn` и изменил внутреннюю структуру `RandomForest`;
- обновился `numpy` и изменил формат массивов;
- обновился Python (например, с 3.10 на 3.12);
— модель может не загрузиться или загрузиться с ошибками.

### Типичные ошибки несовместимости

| Ошибка | Причина |
|--------|---------|
| `ModuleNotFoundError: No module named 'sklearn.ensemble._forest'` | Версия sklearn сильно изменилась, путь к модулю другой |
| `AttributeError: 'RandomForestClassifier' object has no attribute '...'` | Новая версия удалила или переименовала атрибут |
| `ValueError: Buffer dtype mismatch` | Обновился numpy, формат массивов изменился |
| `Can't get attribute 'MyClass'` | Кастомный класс не импортирован при загрузке |

### Как защититься

**1. Фиксируйте версии при сохранении**

In [ ]:
import sklearn
import joblib
import numpy as np
import sys

artifacts = {
    'model': pipeline,
    'versions': {
        'python': sys.version,
        'sklearn': sklearn.__version__,
        'joblib': joblib.__version__,
        'numpy': np.__version__
    }
}
dump(artifacts, 'model_v1.joblib', compress=3)

**2. Проверяйте версии при загрузке**

In [ ]:
import warnings

def load_compatible(path):
    data = load(path)
    versions = data.get('versions', {})
    
    current = {
        'sklearn': sklearn.__version__,
        'joblib': joblib.__version__,
        'numpy': np.__version__
    }
    
    for lib, saved_ver in versions.items():
        if saved_ver != current.get(lib):
            warnings.warn(
                f"{lib}: сохранено с {saved_ver}, текущая {current.get(lib)}"
            )
    
    return data['model']

**3. Используйте `requirements.txt` и виртуальные окружения**

In [ ]:
# Создаём окружение для модели
python -m venv model_env
source model_env/bin/activate
pip install -r requirements.txt  # точные версии из момента сохранения

**4. Docker-контейнеры**

Docker «замораживает» всё окружение. Если модель работала в контейнере — она будет работать всегда, пока не изменится образ.

In [ ]:
FROM python:3.11-slim
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY model.joblib .
COPY main.py .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0"]

## 3. Когда `joblib` не справляется: альтернативы

`joblib` отлично подходит для sklearn-моделей, но не для всего.

### Сравнительная таблица

| Инструмент | Что умеет | Когда использовать вместо joblib |
|------------|-----------|----------------------------------|
| **`dill`** | Сериализует лямбды, функции, классы, которые `pickle` не берёт | Если в Pipeline есть `lambda` или сложные вложенные функции |
| **`cloudpickle`** | Сериализует функции для распределённых вычислений (Spark, Dask, Ray) | Если модель передаётся между процессами/машинами в кластере |
| **`ONNX`** | Кроссплатформенный формат моделей (C++, Java, JS, Python) | Если модель нужно запускать на другом языке или на мобильном устройстве |
| **`safetensors`** | Безопасный формат для весов нейросетей (PyTorch, HuggingFace) | Для нейросетей вместо `torch.save` — быстрее, безопаснее |
| **JSON + `model.get_params()`** | Сохраняет только гиперпараметры | Если веса можно восстановить быстро (например, линейная регрессия с маленькими данными) |

### `dill` — когда нужно сериализовать всё

In [ ]:
import dill

# dill умеет сохранять lambda и вложенные функции
my_func = lambda x: x ** 2

with open('func.pkl', 'wb') as f:
    dill.dump(my_func, f)

with open('func.pkl', 'rb') as f:
    loaded_func = dill.load(f)

print(loaded_func(5))  # 25

> **Но:** `dill` тоже выполняет произвольный код при загрузке. Безопасность не лучше, чем у `joblib`.

### `ONNX` — кроссплатформенный деплой

Если вам нужно запускать модель не в Python, а в мобильном приложении или на сервере на Go:

In [ ]:
# Конвертация sklearn -> ONNX
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('float_input', FloatTensorType([None, 4]))]
onnx_model = convert_sklearn(pipeline, initial_types=initial_type)

with open("model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

Теперь модель работает в C++, Java, JavaScript — без Python и sklearn.

### `safetensors` — для нейросетей

In [ ]:
from safetensors.torch import save_file, load_file
import torch

weights = {"layer1": torch.randn(1000, 1000)}
save_file(weights, "model.safetensors")  # Безопасно, быстро, кроссплатформенно

`joblib` для PyTorch-моделей не используется — это другая экосистема.

### Когда остаёмся на `joblib`

| Сценарий | Почему joblib |
|----------|---------------|
| Классические ML-модели sklearn | Нативная поддержка, оптимизировано под numpy |
| Pipeline с кастомными трансформерами | Сохраняет всю цепочку целиком |
| Быстрый деплой внутри Python-экосистемы | Просто, привычно, не нужны конвертеры |
| Кеширование и параллелизм | Встроено в библиотеку |

## 4. Чек-лист деплоя ML-модели

Перед тем как отправить модель в production, проверьте каждый пункт.

### Чек-лист

- [ ] **Модель сохранена** через `joblib.dump` с `compress=3`
- [ ] **Pipeline** сохранён целиком, а не голая модель
- [ ] **Версии** Python, sklearn, joblib, numpy записаны в метаданные
- [ ] **`requirements.txt`** создан и протестирован в чистом окружении
- [ ] **Хеш файла** (SHA-256) вычислен и сохранён рядом с моделью
- [ ] **Файл модели** загружается в новом окружении без ошибок
- [ ] **Предсказание** на тестовых данных даёт тот же результат, что при обучении
- [ ] **API endpoint** `POST /predict` возвращает ответ за < 100 мс
- [ ] **Endpoint `/health`** показывает, что модель загружена
- [ ] **Batch endpoint** работает корректно на 1000+ строк
- [ ] **Логирование** настроено (загрузка модели, ошибки предсказания)
- [ ] **Мониторинг** метрик качества в production (дрейф данных)
- [ ] **Rollback-план** есть: старая версия модели сохранена и может быть запущена

### Пример проверки перед деплоем

In [ ]:
def validate_model(path):
    """Проверяет модель перед отправкой в production."""
    import hashlib
    
    # 1. Проверяем хеш
    with open(path + '.sha256') as f:
        expected = f.read().strip()
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert expected == actual, "Хеш не совпадает!"
    
    # 2. Загружаем
    model = load(path)
    
    # 3. Проверяем, что это Pipeline
    assert hasattr(model, 'predict'), "Нет метода predict!"
    assert hasattr(model, 'named_steps'), "Не Pipeline!"
    
    # 4. Тестовое предсказание
    import numpy as np
    test_input = np.array([[0, 0, 0, 0]])
    result = model.predict(test_input)
    assert len(result) == 1, "Неверный формат выхода!"
    
    print("✅ Модель прошла валидацию")
    return True

## 5. Итоговая шпаргалка по курсу

### Все инструменты joblib в одном месте

In [ ]:
from joblib import dump, load, Memory, Parallel, delayed

# === СЕРИАЛИЗАЦИЯ ===
dump(model, 'model.joblib', compress=3)          # сохранить
model = load('model.joblib', mmap_mode='r')    # загрузить (mmap — опционально)

# === КЕШИРОВАНИЕ ===
memory = Memory(location='./cache', verbose=1)

@memory.cache
def process(data_path):
    return pd.read_csv(data_path).mean()

# === ПАРАЛЛЕЛИЗМ ===
results = Parallel(n_jobs=-1, verbose=10)(
    delayed(func)(arg) for arg in items
)

### Когда что использовать

| Задача | Инструмент | Параметры |
|--------|-----------|-----------|
| Сохранить модель | `dump` | `compress=3` |
| Загрузить модель быстро | `load` | `mmap_mode='r'` (для numpy) |
| Не пересчитывать предобработку | `Memory` | `location`, `bytes_limit` |
| Перебор гиперпараметров | `Parallel` + `delayed` | `n_jobs=-1`, `backend='loky'` |
| Batch inference | `Parallel` или просто `predict` | порог ~1000 строк |
| Защита от пересчёта в API | `Memory` | кеш внешних запросов |

## 6. Практика: задания

### Задание 7.1: «Проверка хеша»

1. Сохраните модель в `model.joblib`.
2. Напишите функцию `compute_hash(filepath)`, которая возвращает SHA-256 файла.
3. Сохраните хеш в `model.joblib.sha256`.
4. Напишите функцию `load_secure(filepath)`, которая:
   - читает хеш из `.sha256`;
   - сверяет с реальным хешем файла;
   - если совпадает — загружает через `joblib.load`;
   - если нет — выбрасывает `ValueError("File corrupted!")`.
5. Протестируйте: измените один байт в `model.joblib` (откройте в бинарном режиме и замените символ) и убедитесь, что загрузка блокируется.

### Задание 7.2: «Совместимость версий»

1. Создайте словарь артефактов с моделью и версиями библиотек.
2. Сохраните через `joblib.dump`.
3. Напишите функцию `check_compatibility(artifacts)`, которая:
   - сравнивает сохранённые версии с текущими;
   - выводит предупреждение о каждом несовпадении;
   - возвращает `True`, если всё совпадает, и `False`, если есть расхождения.

### Задание 7.3: «Альтернативы»

1. Попробуйте сохранить Pipeline с `lambda` через `joblib.dump` — получите ошибку.
2. Сохраните тот же Pipeline (без `lambda`, с именованной функцией) через `dill.dump`.
3. Загрузите обратно через `dill.load` и убедитесь, что работает.
4. Объясните, почему `dill` — не панацея для production.

### Задание 7.4: «Финальный чек-лист»

1. Возьмите любую модель из предыдущих модулей.
2. Пройдитесь по чек-листу из раздела 4.
3. Запишите результат проверки каждого пункта (пройден / не пройден).
4. Исправьте недочёты, если они есть.

## 7. Эталонные решения

### Решение 7.1

In [ ]:
import hashlib
from joblib import load

def compute_hash(filepath):
    sha256 = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192):
            sha256.update(chunk)
    return sha256.hexdigest()

def load_secure(filepath):
    hash_file = filepath + '.sha256'
    
    with open(hash_file) as f:
        expected = f.read().strip()
    
    actual = compute_hash(filepath)
    
    if expected != actual:
        raise ValueError(
            f"Файл повреждён или изменён!\n"
            f"Ожидалось: {expected}\n"
            f"Получено:  {actual}"
        )
    
    return load(filepath)

# Тест
model = load_secure('model.joblib')

### Решение 7.2

In [ ]:
import sklearn
import joblib
import numpy as np
import warnings

def check_compatibility(artifacts):
    saved = artifacts.get('versions', {})
    current = {
        'sklearn': sklearn.__version__,
        'joblib': joblib.__version__,
        'numpy': np.__version__
    }
    
    ok = True
    for lib, cur_ver in current.items():
        saved_ver = saved.get(lib)
        if saved_ver != cur_ver:
            warnings.warn(
                f"{lib}: сохранено с {saved_ver}, текущая {cur_ver}"
            )
            ok = False
    
    return ok

## 8. Вопросы для самопроверки

1. **Почему `joblib.load()` опасен для файлов из интернета?**  
   *(Ответ: при десериализации может выполниться произвольный Python-код, встроенный в файл. Это называется десериализационной уязвимостью.)*

2. **Как защититься от подмены файла модели?**  
   *(Ответ: вычислять и проверять криптографический хеш файла, например SHA-256. Если хеш не совпадает — не загружать.)*

3. **Почему модель из sklearn 1.3 может не загрузиться в sklearn 1.5?**  
   *(Ответ: внутренняя структура классов, пути импорта или атрибуты могли измениться между версиями.)*

4. **Когда стоит использовать `ONNX` вместо `joblib`?**  
   *(Ответ: когда модель нужно запускать не в Python, а на другом языке (C++, Java, JS) или на мобильном/встроенном устройстве.)*

5. **Почему `dill` не решает проблему безопасности?**  
   *(Ответ: `dill` тоже выполняет код при загрузке, как и `pickle`/`joblib`. Он решает проблему сериализации сложных объектов, но не защищает от вредоносных файлов.)*

6. **Назовите три обязательных пункта чек-листа деплоя.**  
   *(Ответ: сохранение версий библиотек, проверка хеша файла, тестовое предсказание в чистом окружении.)*

## Итоги курса

За 8 модулей мы прошли путь от азов сериализации до production-ready ML-сервиса:

- **Модуль 0** — поняли, что такое сериализация и зачем нужен `joblib`.
- **Модуль 1** — научились сохранять и загружать модели sklearn.
- **Модуль 2** — уменьшали размер файлов через сжатие и memory mapping.
- **Модуль 3** — кешировали результаты функций, чтобы не пересчитывать.
- **Модуль 4** — распараллеливали циклы для ускорения.
- **Модуль 5** — интегрировали `joblib` со sklearn Pipeline.
- **Модуль 6** — встроили модель в FastAPI backend.
- **Модуль 7** — научились защищать модель, проверять совместимость и выбирать инструмент.

**Главное правило:** `joblib` — мощный инструмент, но не универсальный. Используйте его там, где он силён (sklearn, numpy, Python-экосистема), и не бойтесь смотреть в сторону альтернатив, когда задача выходит за рамки.